In [ ]:
# 1. Mount Google Drive.
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# 2. Central path configuration and fresh extraction of the exact codebase archive.
import shutil
from pathlib import Path
from zipfile import ZipFile

DRIVE_DIR = Path("/content/drive/MyDrive/[ICLR] Embedding KD")
ARCHIVE_PATH = DRIVE_DIR / "ICLR_MDD_npa_test_1.zip"
EXTRACT_DIR = Path("/content/ICLR_MDD_npa_test_1_workspace")
STUDENT_MODEL_NAME = "google-bert/bert-base-uncased"
TEACHER_MODEL_NAME = "Qwen/Qwen3-Embedding-4B"

assert DRIVE_DIR.is_dir(), f"Google Drive directory not found: {DRIVE_DIR}"
assert ARCHIVE_PATH.is_file(), f"Codebase archive not found: {ARCHIVE_PATH}"
assert EXTRACT_DIR == Path("/content/ICLR_MDD_npa_test_1_workspace")

# Always start from the code in the selected ZIP, never a stale extracted repo.
if EXTRACT_DIR.is_symlink():
    EXTRACT_DIR.unlink()
elif EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=False)

extract_root = EXTRACT_DIR.resolve()
with ZipFile(ARCHIVE_PATH) as archive:
    unsafe_members = []
    for member in archive.infolist():
        destination = (EXTRACT_DIR / member.filename).resolve()
        if destination != extract_root and extract_root not in destination.parents:
            unsafe_members.append(member.filename)
    assert not unsafe_members, f"Unsafe paths in ZIP: {unsafe_members[:5]}"
    archive.extractall(EXTRACT_DIR)

# Accept a ZIP with or without one enclosing top-level folder.
repo_candidates = sorted(
    {
        main_path.parent.resolve()
        for main_path in EXTRACT_DIR.rglob("main.py")
        if (main_path.parent / "scripts" / "train_heatgeo.sh").is_file()
    },
    key=lambda path: (len(path.parts), str(path)),
)
assert repo_candidates, (
    "The ZIP was extracted, but no HeatGeo repo containing main.py and "
    "scripts/train_heatgeo.sh was found."
)
PROJECT_DIR = repo_candidates[0]
if len(repo_candidates) > 1:
    print(f"Multiple repo roots found; using the shallowest: {PROJECT_DIR}")

# All paths used by the remaining cells are finalized here.
VENV_DIR = PROJECT_DIR / ".venv"
VENV_PYTHON = VENV_DIR / "bin" / "python"
TRAIN_DATA = PROJECT_DIR / "data" / "merged_3_data_5k_each.csv"
HEATGEO_CACHE_DIR = PROJECT_DIR / "cache" / "heatgeo"
RUN_DIR = PROJECT_DIR / "models" / "heatgeo" / "qwen3_4b_to_bert_base"
METRICS_PATH = RUN_DIR / "metrics.jsonl"
LOG_PATH = RUN_DIR / "train.log"
EVALUATION_CSV = RUN_DIR / "evaluation_by_epoch.csv"

assert (PROJECT_DIR / "requirements.txt").is_file(), "requirements.txt is missing"
assert TRAIN_DATA.is_file(), f"Training data not found: {TRAIN_DATA}"

print("Resolved Colab paths:")
for name, path in {
    "archive": ARCHIVE_PATH,
    "extract": EXTRACT_DIR,
    "project": PROJECT_DIR,
    "venv": VENV_DIR,
    "train_data": TRAIN_DATA,
    "run_output": RUN_DIR,
}.items():
    print(f"  {name:12s}: {path}")
print(f"  {'student':12s}: {STUDENT_MODEL_NAME}")
print(f"  {'teacher':12s}: {TEACHER_MODEL_NAME}")

In [ ]:
%cd $PROJECT_DIR

In [ ]:
# 3. Create a clean project-local virtual environment and install packages into it.
import shutil
import subprocess

def run_probe(title, args, *, cwd=None, env=None, allow_failure=False):
    """Run a short command and print its output with an explicit status."""
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)
    result = subprocess.run(
        args,
        cwd=cwd,
        env=env,
        text=True,
        capture_output=True,
    )
    if result.stdout.strip():
        print(result.stdout.rstrip())
    if result.stderr.strip():
        print("[stderr]")
        print(result.stderr.rstrip())
    if result.returncode == 0:
        print(f"[OK] {title}")
    elif allow_failure:
        print(f"[WARN] {title} returned exit code {result.returncode}")
    else:
        raise subprocess.CalledProcessError(
            result.returncode,
            result.args,
            output=result.stdout,
            stderr=result.stderr,
        )
    return result

# Discard any local/macOS venv accidentally stored in the ZIP.
if VENV_DIR.is_symlink():
    VENV_DIR.unlink()
elif VENV_DIR.exists():
    assert PROJECT_DIR.resolve() in VENV_DIR.resolve().parents
    shutil.rmtree(VENV_DIR)

venv_result = run_probe(
    "Create project virtual environment",
    [
        "python3",
        "-m",
        "venv",
        "--system-site-packages",
        "--without-pip",
        str(VENV_DIR),
    ],
    cwd=PROJECT_DIR,
)
assert VENV_PYTHON.is_file(), f"Virtualenv Python was not created: {VENV_PYTHON}"

# --without-pip avoids Colab's ensurepip failure. Pip is inherited from system site-packages.
pip_probe_result = run_probe(
    "Verify pip visible inside project virtual environment",
    [str(VENV_PYTHON), "-m", "pip", "--version"],
    cwd=PROJECT_DIR,
)

print("\n" + "=" * 80)
print("Install requirements (streaming output)")
print("=" * 80)
requirements_result = subprocess.run(
    [
        str(VENV_PYTHON),
        "-m",
        "pip",
        "install",
        "--disable-pip-version-check",
        "-r",
        "requirements.txt",
    ],
    cwd=PROJECT_DIR,
    check=True,
)
print(f"[OK] Requirements installed (exit code {requirements_result.returncode})")

# Fail early if the extracted codebase is not configured for HeatGeo evaluation every epoch.
config_probe = r'''
from config.heatgeo_config import HeatGeoConfig

cfg = HeatGeoConfig()
assert cfg.distill_method == "heatgeo", cfg.distill_method
assert cfg.eval_every == 1, f"Expected eval_every=1, got {cfg.eval_every}"
print(f"HeatGeo config verified: method={cfg.distill_method}, eval_every={cfg.eval_every}")
'''
config_probe_result = run_probe(
    "Verify HeatGeo configuration",
    [str(VENV_PYTHON), "-c", config_probe],
    cwd=PROJECT_DIR,
)
print("\n[READY] Virtual environment and HeatGeo configuration are ready.")

In [ ]:
# 4. Report the GPU that PyTorch and KnowledgeDistiller will actually use.
import shutil
import subprocess

nvidia_smi = shutil.which("nvidia-smi")
if nvidia_smi:
    nvidia_smi_result = run_probe(
        "NVIDIA driver and GPU status (nvidia-smi)",
        [nvidia_smi],
        allow_failure=True,
    )
else:
    nvidia_smi_result = None
    print("\n[WARN] nvidia-smi is unavailable. Select a GPU runtime in Colab.")

device_probe = r'''
import sys
import torch

cuda_count = torch.cuda.device_count()
mps_available = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
if cuda_count >= 2:
    selected = "student=cuda:0, teacher=cuda:1"
elif torch.cuda.is_available():
    selected = "student=cuda:0, teacher=cuda:0"
elif mps_available:
    selected = "student=mps, teacher=mps"
else:
    selected = "student=cpu, teacher=cpu"

print(f"Python environment: {sys.executable}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()} (device count={cuda_count})")
if cuda_count:
    for index in range(cuda_count):
        print(f"  cuda:{index}: {torch.cuda.get_device_name(index)}")
print(f"MPS available: {mps_available}")
print(f"KnowledgeDistiller will use: {selected}")
if torch.cuda.is_available():
    print("GPU STATUS: READY - HeatGeo training will use CUDA.")
else:
    print("GPU STATUS: NOT USING CUDA - enable a Colab GPU runtime before training.")
'''
device_probe_result = run_probe(
    "PyTorch accelerator selection",
    [str(VENV_PYTHON), "-c", device_probe],
    cwd=PROJECT_DIR,
)
print("\n[READY] Accelerator check completed.")

In [ ]:
# 5. Remove only stale HeatGeo cache and this reproduction run's output.
import shutil

for path in (HEATGEO_CACHE_DIR, RUN_DIR):
    resolved = path.resolve()
    assert PROJECT_DIR.resolve() in resolved.parents, f"Unsafe cleanup target: {resolved}"
    if path.is_symlink():
        path.unlink()
        print(f"Removed stale symlink: {path}")
    elif resolved.exists():
        shutil.rmtree(resolved)
        print(f"Removed stale artifacts: {resolved}")

RUN_DIR.mkdir(parents=True, exist_ok=False)
print(f"Clean run output: {RUN_DIR}")

In [ ]:
# 6. Train with HeatGeo. Validation runs after every epoch; test runs only at the end.
import os
import subprocess

training_env = os.environ.copy()
training_env.update(
    {
        "TRAIN_DATA": str(TRAIN_DATA),
        "STUDENT_MODEL": STUDENT_MODEL_NAME,
        "TEACHER_MODEL": TEACHER_MODEL_NAME,
        "BATCH_SIZE": "4",
        "EPOCHS": "5",
        "LR": "2e-5",
        "MAX_LENGTH": "256",
        "SAVE_DIR": str(RUN_DIR),
        "PYTHONUNBUFFERED": "1",
        "WANDB_MODE": "disabled",
    }
)

print("\n" + "=" * 80)
print("HeatGeo training configuration")
print("=" * 80)
print(f"Student model : {STUDENT_MODEL_NAME}")
print(f"Teacher model : {TEACHER_MODEL_NAME}")
print(f"Training data : {TRAIN_DATA}")
print(f"Batch size    : {training_env['BATCH_SIZE']}")
print(f"Epochs        : {training_env['EPOCHS']}")
print(f"Learning rate : {training_env['LR']}")
print(f"Output        : {RUN_DIR}")
print(f"Log           : {LOG_PATH}")
print("=" * 80)
print("Training output will stream below.\n")

train_command = f'''
set -o pipefail
source "{VENV_DIR}/bin/activate"
bash scripts/train_heatgeo.sh --no_wandb 2>&1 | tee "{LOG_PATH}"
'''
train_result = subprocess.run(
    ["bash", "-lc", train_command],
    cwd=PROJECT_DIR,
    env=training_env,
    check=True,
)

print(f"\n[OK] HeatGeo training command finished (exit code {train_result.returncode})")
assert METRICS_PATH.is_file(), f"Training finished without metrics: {METRICS_PATH}"
print(f"[OK] Metrics file found: {METRICS_PATH}")
print(f"[OK] Full training log: {LOG_PATH}")

In [ ]:
# 7. Build one general validation table with benchmark and family-mean rows per epoch.
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

METRIC_COLUMNS = [
    "accuracy",
    "f1",
    "precision",
    "recall",
    "average_precision",
    "spearman",
]

def benchmark_name(path, split):
    name = Path(path).stem
    suffix = f"_{split}"
    return name[:-len(suffix)] if name.endswith(suffix) else name

def validation_rows(record):
    validation = record.get("validation")
    epoch = record.get("train", {}).get("epoch")
    if not validation or epoch is None:
        return []

    rows = []
    for family in ("classification", "pair", "sts"):
        for path, result in validation.get(family, {}).items():
            row = {
                "epoch": int(epoch),
                "family": family,
                "benchmark": benchmark_name(path, "validation"),
            }
            if family == "sts":
                row["spearman"] = float(result)
            else:
                row.update(
                    {
                        metric: float(value)
                        for metric, value in result.items()
                        if metric in METRIC_COLUMNS
                    }
                )
            rows.append(row)
    return rows

with METRICS_PATH.open(encoding="utf-8") as handle:
    records = [json.loads(line) for line in handle if line.strip()]

detail_rows = [row for record in records for row in validation_rows(record)]
assert detail_rows, f"No per-epoch validation records found in {METRICS_PATH}"

detail_table = pd.DataFrame(detail_rows).reindex(
    columns=["epoch", "family", "benchmark", *METRIC_COLUMNS]
)
mean_table = (
    detail_table.groupby(["epoch", "family"], as_index=False)[METRIC_COLUMNS]
    .mean()
    .assign(benchmark="MEAN")
)
evaluation_by_epoch = pd.concat([detail_table, mean_table], ignore_index=True)

family_order = {"classification": 0, "pair": 1, "sts": 2}
evaluation_by_epoch["_family_order"] = evaluation_by_epoch["family"].map(family_order)
evaluation_by_epoch["_mean_order"] = evaluation_by_epoch["benchmark"].eq("MEAN")
evaluation_by_epoch = (
    evaluation_by_epoch.sort_values(
        ["epoch", "_family_order", "_mean_order", "benchmark"]
    )
    .drop(columns=["_family_order", "_mean_order"])
    .reset_index(drop=True)
)

evaluation_by_epoch.to_csv(EVALUATION_CSV, index=False)
display(evaluation_by_epoch.style.format(precision=4, na_rep="—"))
print(f"Saved evaluation table: {EVALUATION_CSV}")